# NB5 · Stima dei volumi quando i dati sono pochissimi

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/battles5/conad-statistical-learning/blob/main/notebooks/NB5_caso_fontanelli.ipynb)

**il caso fontanelli: 3 negozi misurati, 500 da stimare, e perché qui il machine learning non basta**

corso *Apprendimento statistico* per CONAD Nord Ovest · Orso Peruzzi (IFAB)

> come si esegue una cella: clicca dentro e premi **Shift + Invio**; esegui le celle in ordine, dall'alto verso il basso.

un'insegna del Nord Ovest vuole installare **fontanelli** (erogatori d'acqua) in 500 negozi e stimare quanta acqua erogheranno. ma ne ha installati solo in **3**. la domanda: quanti litri a settimana aspettarsi sull'intera rete?

è il caso limite del mattino: **pochissime osservazioni** ($n = 3$) e più variabili che dati.

> nota didattica: il dataset è sintetico e contiene di nascosto la verità per **tutti** i negozi, così alla fine possiamo svelare il totale vero e giudicare ogni approccio. nella realtà l'azienda ha solo i 3 pilota.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def carica(nome):
    "carica dal file locale ../data se presente (sviluppo), altrimenti dal repo GitHub (Colab)"
    locale = os.path.join("..", "data", nome)
    if os.path.exists(locale):
        return pd.read_csv(locale)
    base = "https://raw.githubusercontent.com/battles5/conad-statistical-learning/main/data/"
    return pd.read_csv(base + nome)


## 1. quello che sappiamo: 3 negozi

`misurato = 1` sono i 3 negozi con il fontanello già installato (uno a Prato, uno in Liguria, uno nel Lazio). sono gli unici dati veri.

In [ ]:
df = carica("fontanelli_negozi.csv")
pilota = df[df["misurato"] == 1].copy()
da_stimare = df[df["misurato"] == 0].copy()
print("negozi totali:", len(df), " | misurati:", len(pilota), " | da stimare:", len(da_stimare))

# il tasso per visita dei 3 pilota: quanto varia?
pilota["litri_per_visita"] = pilota["litri_erogati_sett"] / (pilota["visite_giorno"] * 7)
pilota[["negozio_id", "regione", "tipologia", "visite_giorno", "litri_erogati_sett", "litri_per_visita"]].round(4)

già qui un campanello d'allarme: il **tasso per visita** dei 3 negozi varia parecchio. con soli 3 punti non sappiamo quale sia quello "giusto".

## 2. la tentazione: buttiamoci un modello

proviamo comunque ad addestrare una regressione lineare sui 3 negozi.

> **indovina prima di eseguire**: che $R^2$ farà sul training con 3 punti e più variabili?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# poche variabili di base, stesse colonne per tutti (modelli allineati)
Xall = pd.get_dummies(
    df[["visite_giorno", "acqua_bottiglia_litri_sett", "temp_media_zona", "tipologia"]],
    columns=["tipologia"], drop_first=True,
)
Xp = Xall[df["misurato"] == 1]
Xt = Xall[df["misurato"] == 0]
yp = df.loc[df["misurato"] == 1, "litri_erogati_sett"]

modello = LinearRegression().fit(Xp, yp)
print("R2 sul training (3 negozi):", round(r2_score(yp, modello.predict(Xp)), 3))

pred = modello.predict(Xt)
print("previsioni sui 500 negozi:")
print(f"  minima: {pred.min():.0f} litri/sett   massima: {pred.max():.0f} litri/sett")
print(f"  quante previsioni NEGATIVE (impossibili): {int((pred < 0).sum())} su {len(pred)}")
print(f"  totale 'stimato' dal modello: {pred.sum():.0f} litri/sett")

$R^2 = 1{,}000$: il modello passa **perfetto** per i 3 punti. ma è una trappola: con più variabili che dati il sistema è sottodeterminato. infatti sui 500 negozi sputa previsioni assurde, perfino **negative**. è overfitting puro: ha "memorizzato" i 3 negozi e non sa generalizzare.

## 3. la stima onesta: un tasso, con incertezza

niente modello: stimiamo un **tasso** dai 3 pilota e lo applichiamo ai 500, dichiarando l'incertezza.

> **manopola**: cambia `DRIVER` tra `"visite_giorno"` e `"acqua_bottiglia_litri_sett"` e guarda quale dà una banda meno larga.

In [ ]:
# >>> MANOPOLA: la variabile su cui basare il tasso <<<
DRIVER = "visite_giorno"

fattore = 7 if DRIVER == "visite_giorno" else 1   # visite/giorno -> visite/settimana
base_p = pilota[DRIVER].values * fattore
k = pilota["litri_erogati_sett"].values / base_p
base_t = da_stimare[DRIVER].values * fattore

stima_punto = (base_t * k.mean()).sum()
stima_bassa = (base_t * k.min()).sum()
stima_alta = (base_t * k.max()).sum()

print(f"driver: {DRIVER}")
print(f"tassi dei 3 pilota: {np.round(k, 4)}")
print(f"stima dei 500 negozi (tasso medio): {stima_punto:,.0f} litri/sett")
print(f"banda di incertezza (min-max tasso): {stima_bassa:,.0f}  -  {stima_alta:,.0f} litri/sett")

la stima puntuale dà un ordine di grandezza, ma la **banda è larga**: con 3 negozi non possiamo essere precisi, e dirlo è parte del lavoro.

## 4. e se misurassimo più negozi?

la vera soluzione non è statistica, è raccogliere dati: simuliamo un **pilota da 30 negozi**. ora una semplice regressione funziona.

In [ ]:
from sklearn.model_selection import train_test_split

noti = da_stimare.sample(30, random_state=1)          # i 30 che misuriamo
resto = da_stimare.drop(noti.index)                    # gli altri 470, da prevedere
Xn = Xall.loc[noti.index]; yn = noti["litri_erogati_sett"]

# stima onesta dell'errore con un train/test interno ai 30
Xtr, Xte, ytr, yte = train_test_split(Xn, yn, test_size=10, random_state=0)
reg = LinearRegression().fit(Xtr, ytr)
print("R2 su negozi nuovi (test interno ai 30):", round(r2_score(yte, reg.predict(Xte)), 3))

# rifit su tutti i 30 e stima del totale dei 500 = 30 misurati + 470 previsti
reg_full = LinearRegression().fit(Xn, yn)
pred_resto = np.clip(reg_full.predict(Xall.loc[resto.index]), 0, None)
stima_reg = yn.sum() + pred_resto.sum()
print(f"stima dei 500 negozi (pilota da 30): {stima_reg:,.0f} litri/sett")

con 30 negozi ben scelti l'$R^2$ sui negozi nuovi diventa decente e la stima si stringe: ora **c'è un vero train/test**.

## 5. il verdetto

sveliamo la verità nascosta e confrontiamo i tre approcci sul totale dei 500 negozi, guardando sia il totale sia l'errore **per negozio**.

In [ ]:
from sklearn.metrics import mean_absolute_error

vero = da_stimare["litri_erogati_sett"].values
vero_totale = vero.sum()

stime = {
    "modello sui 3 negozi": pred,
    "tasso (stima puntuale)": base_t * k.mean(),
    "regressione su 30 negozi": np.clip(reg_full.predict(Xt), 0, None),
}
righe = []
for nome, s in stime.items():
    righe.append({
        "metodo": nome,
        "totale_litri_sett": round(float(np.sum(s))),
        "errore_totale_%": round(100 * (np.sum(s) - vero_totale) / vero_totale, 1),
        "errore_per_negozio_litri": round(mean_absolute_error(vero, s)),
    })
riassunto = pd.DataFrame(righe)
print(f"TOTALE VERO dei 500 negozi: {vero_totale:,.0f} litri/sett\n")
print(riassunto.to_string(index=False))
print(f"\n(il modello sui 3 negozi produce anche {int((pred < 0).sum())} previsioni negative, impossibili)")

attenzione a non farsi ingannare dal **totale**: il modello sui 3 negozi può azzeccarlo per puro caso, perché gli errori enormi sui singoli negozi si compensano nella somma. l'**errore per negozio** lo smaschera, ed è quello che conta per decidere capienza e priorità negozio per negozio.

morale: con $n$ piccolissimo nessun modello flessibile aiuta; servono ipotesi forti più incertezza, oppure un **pilota più ampio**.

---

# Parte 2 · più variabili, e come trattarle

ora che il pilota è più ampio (30 negozi) possiamo permetterci più variabili. ma attenzione: **come** prepari le variabili conta più del modello. vediamo la geografia e le altre, una per una.

In [ ]:
print("colonne disponibili:", list(df.columns))
pilota[["negozio_id", "regione", "tipologia", "superficie_mq",
        "presenza_area_ristoro", "giorni_caldi_anno", "temp_media_zona"]]

## la geografia: non one-hot delle 6 regioni

l'azienda copre **6 regioni** del Nord Ovest, dalla Valle d'Aosta alla Sardegna. la tentazione è mettere la `regione` così com'è (one-hot): ma sono **6 categorie**, cioè 5 colonne nuove. con pochi negozi misurati ogni regione avrebbe pochissimi dati, e le stime per regione sarebbero ballerine (varianza alta), oltre al rischio di regioni **assenti** dal pilota.

meglio **raggruppare** le regioni in poche **macro-aree** per clima: meno colonne, più dati per gruppo, meno varianza. il filo del mattino: ridurre la flessibilità quando i dati sono pochi.

In [ ]:
# regione -> macro-area climatica (3 gruppi invece di 6)
macro = {
    "Valle d'Aosta": "nord", "Piemonte": "nord",
    "Liguria": "centro", "Toscana": "centro",
    "Lazio": "centro-sud", "Sardegna": "centro-sud",
}
df["macro_area"] = df["regione"].map(macro)
print(df["macro_area"].value_counts().to_dict())
print("\nin alternativa, il numero dietro la geografia: 'giorni_caldi_anno'")
print("è il driver fisico del consumo, e basta una colonna invece di tante dummy.")

## le altre variabili, e come le tratto

- **macro_area** (da `regione`): one-hot delle 3 macro-aree (2 colonne); cattura il clima senza esplodere in dummy;
- **presenza_area_ristoro** (0/1): se c'è un bar la gente sosta e riempie le borracce, è un driver vero; la lascio binaria;
- **superficie_mq**: dimensione del negozio, numerica, la tengo così;
- **visite_giorno**: l'afflusso, il driver principale, numerica;
- **tipologia**: prossimità o attrazione, one-hot (2 livelli, 1 colonna);
- **temp_media_zona**: è una lettura locale **rumorosa**; la lascio **fuori**, perché la macro-area dà lo stesso segnale in modo più stabile;
- **acqua_bottiglia_litri_sett**: è un proxy molto **correlato** alle visite (collinearità); la lascio fuori per non duplicare lo stesso segnale.

ricostruiamo il modello sui 30 negozi con queste variabili trattate bene, e confrontiamo con quello "povero" della Parte 1.

In [ ]:
def features_ricche(d):
    base = d[["visite_giorno", "superficie_mq", "presenza_area_ristoro"]].astype(float)
    area = pd.get_dummies(d["macro_area"], prefix="area", drop_first=True).astype(float)
    tipo = pd.get_dummies(d["tipologia"], drop_first=True).astype(float)
    return pd.concat([base, area, tipo], axis=1)

Xrich = features_ricche(df)

# stesse 30 misurate di prima, ma con le variabili ricche
reg_rich = LinearRegression().fit(Xrich.loc[noti.index], yn)
pred_rich = np.clip(reg_rich.predict(Xrich.loc[da_stimare.index]), 0, None)
pred_base = np.clip(reg_full.predict(Xall.loc[da_stimare.index]), 0, None)

# R2 onesto su negozi nuovi (test interno ai 30)
Xrtr, Xrte, yrtr, yrte = train_test_split(Xrich.loc[noti.index], yn, test_size=10, random_state=0)
r2_rich = r2_score(yrte, LinearRegression().fit(Xrtr, yrtr).predict(Xrte))

print("--- errore PER NEGOZIO (litri/sett) ---")
print("modello povero (Parte 1):     ", round(mean_absolute_error(vero, pred_base)))
print("modello ricco (Parte 2):      ", round(mean_absolute_error(vero, pred_rich)))
print("\n--- R2 su negozi nuovi ---")
print("modello ricco (Parte 2):      ", round(r2_rich, 3))
print("\n--- totale dei 500 ---")
print("stima ricca:", f"{pred_rich.sum():,.0f}", " | vero:", f"{vero_totale:,.0f}",
      " | errore:", f"{100*(pred_rich.sum()-vero_totale)/vero_totale:+.1f}%")

trattare bene le variabili (raggruppare la geografia, aggiungere i driver veri come l'area ristoro, scartare i proxy rumorosi o ridondanti) **riduce l'errore per negozio** rispetto al modello povero, a parità di dati. non è più flessibilità: è **feature engineering ragionato**, guidato dal buon senso sul fenomeno.

il messaggio per il cliente: con un pilota di qualche decina di negozi e poche variabili scelte bene, la stima per la rete diventa affidabile; con 3 negozi, no.

---

### cella bonus: la cross-validation con 3 punti

leave-one-out sui 3 pilota: alleniamo il tasso su 2 negozi e proviamo sul terzo, a rotazione. si vede perché con $n = 3$ la validazione non è affidabile.

In [ ]:
# BONUS
litri = pilota["litri_erogati_sett"].values
visite_sett = pilota["visite_giorno"].values * 7
for i in range(3):
    altri = [j for j in range(3) if j != i]
    k_due = (litri[altri] / visite_sett[altri]).mean()      # tasso dai due "noti"
    prev = visite_sett[i] * k_due                            # previsione sul terzo
    v = litri[i]
    print(f"negozio escluso {pilota['negozio_id'].iloc[i]}: previsto {prev:.0f}, vero {v:.0f}, "
          f"errore {100*(prev-v)/v:+.0f}%")